In [ ]:
import pandas as pd
import joblib

In [ ]:
df = pd.read_csv("../data_processed/transit_2026_prepared.csv")

df["timestamp"] = pd.to_datetime(
    df["sd_date"].astype(str) + " " +
    df["st_time"].astype(str)
)

df = df.sort_values(["object_key", "timestamp"])

In [ ]:
model_8h = joblib.load("../models/model_8h.pkl")

In [ ]:
features = [
    "distance_mtr",
    "duration_sec",
    "stay_duration_sec",
    "segment_count",
    "distance_floor",
    "hour",
    "time_period",
    "movement_ratio",
    "avg_speed",
    "activity_level",
    "sl_floor_local_name"
]

In [ ]:
def classify_probability(p):
    if p < 0.3:
        return "low"
    elif p < 0.7:
        return "medium"
    else:
        return "high"

In [ ]:
def predict_device_group(object_key):
    
    device_data = df[df["object_key"] == object_key]
    
    if device_data.empty:
        return "Device not found"
    
    # Seneste observation
    latest = device_data.sort_values("timestamp").iloc[-1:]
    
    X_input = latest[features]
    
    prob = model_8h.predict_proba(X_input)[:, 1][0]
    
    group = classify_probability(prob)
    
    return {
        "object_key": object_key,
        "timestamp": latest["timestamp"].values[0],
        "probability": prob,
        "group": group
    }

In [ ]:
# vælg en rigtig device
object_key = df["object_key"].drop_duplicates().iloc[0]

result = predict_device_group(object_key)

print(result)

In [ ]:
def interpret_group(group):
    
    if group == "low":
        return "Likely inactive – candidate for reallocation"
    
    elif group == "medium":
        return "Uncertain – monitor"
    
    else:
        return "Likely active – do not move"

In [ ]:
object_key = 163655

if object_key not in df["object_key"].values:
    print("Device not found")
else:
    result = predict_device_group(object_key)
    
    print("\nDevice:", result["object_key"])
    print("Probability:", round(result["probability"], 3))
    print("Group:", result["group"])
    print("Decision:", interpret_group(result["group"]))

## With UI

In [ ]:
from IPython.display import display
import pandas as pd

In [ ]:
def display_result_ui(result):
    
    prob = f"{result['probability'] * 100:.1f}%"
    group_raw = result["group"]
    group = group_raw.upper()
    decision = interpret_group(group_raw)
    
    timestamp_clean = pd.to_datetime(result["timestamp"]).strftime("%Y-%m-%d %H:%M")
    
    df_display = pd.DataFrame({
        "Metric": [
            "Device ID",
            "Last observed",
            "Probability (8h)",
            "Category",
            "Decision"
        ],
        "Value": [
            result["object_key"],
            timestamp_clean,
            prob,
            group,
            decision
        ]
    })
    
    # 🔹 Simple color mapping
    def color_category(val):
        if val == "LOW":
            return "color: #2e7d32; font-weight: bold;"
        elif val == "MEDIUM":
            return "color: #ef6c00; font-weight: bold;"
        elif val == "HIGH":
            return "color: #c62828; font-weight: bold;"
        return ""
    
    styled = (
        df_display.style
        .hide(axis="index")
        .set_properties(**{
            "text-align": "left",
            "padding": "6px"
        })
        .set_table_styles([
            {"selector": "th", "props": [
                ("text-align", "left"),
                ("border-bottom", "1px solid #ccc")
            ]}
        ])
        .map(color_category, subset=pd.IndexSlice[3, "Value"])
    )
    
    display(styled)

In [ ]:
object_key = 201472  # <- Device ID

if object_key not in df["object_key"].values:
    print(f"Device {object_key} not found")
else:
    result = predict_device_group(object_key)
    display_result_ui(result)

## Example

In [ ]:
201472

In [ ]:
# Get latest observation per device
latest_df = (
    df.sort_values("timestamp")
    .groupby("object_key")
    .tail(1)
)

# Predict probabilities
latest_df["probability"] = model_8h.predict_proba(latest_df[features])[:, 1]

# Classify
latest_df["group"] = latest_df["probability"].apply(classify_probability)

In [ ]:
low_device = latest_df[latest_df["group"] == "low"]["object_key"].iloc[0]
medium_device = latest_df[latest_df["group"] == "medium"]["object_key"].iloc[0]
high_device = latest_df[latest_df["group"] == "high"]["object_key"].iloc[0]

print("LOW:", low_device)
print("MEDIUM:", medium_device)
print("HIGH:", high_device)

In [ ]:
from IPython.display import display
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ---------------------------
# Config
# ---------------------------
cmap = mpl.colormaps["viridis"]

color_map = {"LOW": 0.15, "MEDIUM": 0.5, "HIGH": 0.85}

decision_map = {
    "low": "Likely inactive → reallocate",
    "medium": "Uncertain → monitor",
    "high": "Likely active → keep"
}

# ---------------------------
# Build table
# ---------------------------
df_cases = pd.DataFrame([
    {
        "Case": label,
        "Device ID": r["object_key"],
        "Last observed": pd.to_datetime(r["timestamp"]).strftime("%Y-%m-%d %H:%M"),
        "Probability (8h)": f"{r['probability']:.2f}",
        "Category": r["group"].upper(),
        "Decision": decision_map[r["group"]]
    }
    for label, obj in devices.items()
    for r in [predict_device_group(obj)]
]).set_index("Case").T


# ---------------------------
# Notebook styling
# ---------------------------
def viridis(val):
    if val in color_map:
        c = mpl.colors.to_hex(cmap(color_map[val]))
        return f"background-color: {c}; color: white;"
    return ""

styled = (
    df_cases.style
    .map(viridis, subset=pd.IndexSlice["Category", :])
    .set_properties(**{"text-align": "center", "font-size": "11pt"})
    .set_table_styles([
        {"selector": "th", "props": [("font-weight", "bold"), ("border-bottom", "1px solid #ccc")]},
        {"selector": "td", "props": [("border", "1px solid #eee")]}
    ])
)

display(styled)


# ---------------------------
# PNG export (matplotlib)
# ---------------------------
fig, ax = plt.subplots(figsize=(14, 4))
ax.axis('off')

table = ax.table(
    cellText=df_cases.values,
    rowLabels=df_cases.index,
    colLabels=df_cases.columns,
    loc='center',
    cellLoc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

# Style + colors
cat_idx = list(df_cases.index).index("Category")

for (r, c), cell in table.get_celld().items():
    if r == 0 or c == -1:
        cell.set_text_props(weight='bold')
    cell.set_edgecolor("#eeeeee")

for c, val in enumerate(df_cases.iloc[cat_idx]):
    if val in color_map:
        cell = table[(cat_idx + 1, c)]
        cell.set_facecolor(cmap(color_map[val]))
        cell.get_text().set_color("white")

plt.savefig("figure_6_activity_cases.png", dpi=300, bbox_inches='tight')
plt.close()

print("Saved as figure_6_activity_cases.png")